In [3]:
# 1-dataset model (HTCas9)

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_HT11():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_HT11():
    with open('HT1-1_indel_frequency_value_percentage_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_1_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_HT11 = load_branch1_data_HT11()
    X1_HT11   = np.asarray(X1_HT11)
    X1 = np.concatenate([X1_HT11], axis=0) 

    rates_HT11 = load_reaction_rates_HT11()
    rates_HT11   = np.asarray(rates_HT11)
    rates = np.concatenate([rates_HT11], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen HT11 dataset
    np.random.seed(42)
    full_indices_HT11 = np.arange(len(rates_HT11))
    selected_indices_HT11 = np.random.choice(len(full_indices_HT11), size=len(full_indices_HT11), replace=False)
    unseen_indices_HT11 = np.setdiff1d(full_indices_HT11, selected_indices_HT11)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_HT11 = Subset(hybrid_dataset, unseen_indices_HT11)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_HT11 = Subset(hybrid_dataset, selected_indices_HT11)
    trial_loader = DataLoader(selected_set_HT11, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_1_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_1_dataset_trained_CNN1.pt: Spearman = 0.2566573931166369
../../../CNN1_only_models/trial_1_best_1_dataset_trained_CNN1.pt: Spearman = 0.2876440566295282
../../../CNN1_only_models/trial_2_best_1_dataset_trained_CNN1.pt: Spearman = 0.27693446999925264
../../../CNN1_only_models/trial_3_best_1_dataset_trained_CNN1.pt: Spearman = 0.24830526343582662
../../../CNN1_only_models/trial_4_best_1_dataset_trained_CNN1.pt: Spearman = 0.25000607334221164
../../../CNN1_only_models/trial_5_best_1_dataset_trained_CNN1.pt: Spearman = 0.26457806183999383
../../../CNN1_only_models/trial_6_best_1_dataset_trained_CNN1.pt: Spearman = 0.23923530022322098
../../../CNN1_only_models/trial_7_best_1_dataset_trained_CNN1.pt: Spearman = 0.28022894911867785
../../../CNN1_only_models/trial_8_best_1_dataset_trained_CNN1.pt: Spearman = 0.25809971615721766
../../../CNN1_only_models/trial_9_best_1_dataset_trained_CNN1.pt: Spearman = 0.255356002206604


In [5]:
# 2-dataset model (HTCas9+HT11)

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_HT11():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_HT11():
    with open('HT1-1_indel_frequency_value_percentage_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_2_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_HT11 = load_branch1_data_HT11()
    X1_HT11   = np.asarray(X1_HT11)
    X1 = np.concatenate([X1_HT11], axis=0) 

    rates_HT11 = load_reaction_rates_HT11()
    rates_HT11   = np.asarray(rates_HT11)
    rates = np.concatenate([rates_HT11], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen HT11 dataset
    np.random.seed(42)
    full_indices_HT11 = np.arange(len(rates_HT11))
    selected_indices_HT11 = np.random.choice(len(full_indices_HT11), size=len(full_indices_HT11), replace=False)
    unseen_indices_HT11 = np.setdiff1d(full_indices_HT11, selected_indices_HT11)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_HT11 = Subset(hybrid_dataset, unseen_indices_HT11)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_HT11 = Subset(hybrid_dataset, selected_indices_HT11)
    trial_loader = DataLoader(selected_set_HT11, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_2_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_2_dataset_trained_CNN1.pt: Spearman = 0.6051484562569651
../../../CNN1_only_models/trial_1_best_2_dataset_trained_CNN1.pt: Spearman = 0.5100104451305524
../../../CNN1_only_models/trial_2_best_2_dataset_trained_CNN1.pt: Spearman = 0.52293221887488
../../../CNN1_only_models/trial_3_best_2_dataset_trained_CNN1.pt: Spearman = 0.5186030999002745
../../../CNN1_only_models/trial_4_best_2_dataset_trained_CNN1.pt: Spearman = 0.5494498530261567
../../../CNN1_only_models/trial_5_best_2_dataset_trained_CNN1.pt: Spearman = 0.4500906134170019
../../../CNN1_only_models/trial_6_best_2_dataset_trained_CNN1.pt: Spearman = 0.5323729132227547
../../../CNN1_only_models/trial_7_best_2_dataset_trained_CNN1.pt: Spearman = 0.5321977641651346
../../../CNN1_only_models/trial_8_best_2_dataset_trained_CNN1.pt: Spearman = 0.42798873866130416
../../../CNN1_only_models/trial_9_best_2_dataset_trained_CNN1.pt: Spearman = 0.48780286857371047


In [7]:
# 4-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5))

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_HT11():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_HT11():
    with open('HT1-1_indel_frequency_value_percentage_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_4_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_HT11 = load_branch1_data_HT11()
    X1_HT11   = np.asarray(X1_HT11)
    X1 = np.concatenate([X1_HT11], axis=0) 

    rates_HT11 = load_reaction_rates_HT11()
    rates_HT11   = np.asarray(rates_HT11)
    rates = np.concatenate([rates_HT11], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen HT11 dataset
    np.random.seed(42)
    full_indices_HT11 = np.arange(len(rates_HT11))
    selected_indices_HT11 = np.random.choice(len(full_indices_HT11), size=len(full_indices_HT11), replace=False)
    unseen_indices_HT11 = np.setdiff1d(full_indices_HT11, selected_indices_HT11)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_HT11 = Subset(hybrid_dataset, unseen_indices_HT11)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_HT11 = Subset(hybrid_dataset, selected_indices_HT11)
    trial_loader = DataLoader(selected_set_HT11, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_4_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_4_dataset_trained_CNN1.pt: Spearman = 0.5102884605980191
../../../CNN1_only_models/trial_1_best_4_dataset_trained_CNN1.pt: Spearman = 0.5025357821912717
../../../CNN1_only_models/trial_2_best_4_dataset_trained_CNN1.pt: Spearman = 0.5706029040792527
../../../CNN1_only_models/trial_3_best_4_dataset_trained_CNN1.pt: Spearman = 0.5385973905065291
../../../CNN1_only_models/trial_4_best_4_dataset_trained_CNN1.pt: Spearman = 0.5153091791352241
../../../CNN1_only_models/trial_5_best_4_dataset_trained_CNN1.pt: Spearman = 0.4434962164767413
../../../CNN1_only_models/trial_6_best_4_dataset_trained_CNN1.pt: Spearman = 0.4195766374909648
../../../CNN1_only_models/trial_7_best_4_dataset_trained_CNN1.pt: Spearman = 0.5458588471677903
../../../CNN1_only_models/trial_8_best_4_dataset_trained_CNN1.pt: Spearman = 0.5811806000672084
../../../CNN1_only_models/trial_9_best_4_dataset_trained_CNN1.pt: Spearman = 0.5296587489791391


In [9]:
# 5-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5)+CHANGE_seq)

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_HT11():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_HT11():
    with open('HT1-1_indel_frequency_value_percentage_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_5_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_HT11 = load_branch1_data_HT11()
    X1_HT11   = np.asarray(X1_HT11)
    X1 = np.concatenate([X1_HT11], axis=0) 

    rates_HT11 = load_reaction_rates_HT11()
    rates_HT11   = np.asarray(rates_HT11)
    rates = np.concatenate([rates_HT11], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen HT11 dataset
    np.random.seed(42)
    full_indices_HT11 = np.arange(len(rates_HT11))
    selected_indices_HT11 = np.random.choice(len(full_indices_HT11), size=len(full_indices_HT11), replace=False)
    unseen_indices_HT11 = np.setdiff1d(full_indices_HT11, selected_indices_HT11)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_HT11 = Subset(hybrid_dataset, unseen_indices_HT11)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_HT11 = Subset(hybrid_dataset, selected_indices_HT11)
    trial_loader = DataLoader(selected_set_HT11, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_5_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_5_dataset_trained_CNN1.pt: Spearman = 0.5490271126118699
../../../CNN1_only_models/trial_1_best_5_dataset_trained_CNN1.pt: Spearman = 0.5821640057217026
../../../CNN1_only_models/trial_2_best_5_dataset_trained_CNN1.pt: Spearman = 0.5462744468581663
../../../CNN1_only_models/trial_3_best_5_dataset_trained_CNN1.pt: Spearman = 0.4870785206740738
../../../CNN1_only_models/trial_4_best_5_dataset_trained_CNN1.pt: Spearman = 0.534259920892688
../../../CNN1_only_models/trial_5_best_5_dataset_trained_CNN1.pt: Spearman = 0.5156590095135293
../../../CNN1_only_models/trial_6_best_5_dataset_trained_CNN1.pt: Spearman = 0.45760748734209883
../../../CNN1_only_models/trial_7_best_5_dataset_trained_CNN1.pt: Spearman = 0.468276593455964
../../../CNN1_only_models/trial_8_best_5_dataset_trained_CNN1.pt: Spearman = 0.38099408842668125
../../../CNN1_only_models/trial_9_best_5_dataset_trained_CNN1.pt: Spearman = 0.43504282955982354


In [11]:
# 6-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5)+CHANGE_seq+TIGER)

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_HT11():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_HT11():
    with open('HT1-1_indel_frequency_value_percentage_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_6_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_HT11 = load_branch1_data_HT11()
    X1_HT11   = np.asarray(X1_HT11)
    X1 = np.concatenate([X1_HT11], axis=0) 

    rates_HT11 = load_reaction_rates_HT11()
    rates_HT11   = np.asarray(rates_HT11)
    rates = np.concatenate([rates_HT11], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen HT11 dataset
    np.random.seed(42)
    full_indices_HT11 = np.arange(len(rates_HT11))
    selected_indices_HT11 = np.random.choice(len(full_indices_HT11), size=len(full_indices_HT11), replace=False)
    unseen_indices_HT11 = np.setdiff1d(full_indices_HT11, selected_indices_HT11)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_HT11 = Subset(hybrid_dataset, unseen_indices_HT11)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_HT11 = Subset(hybrid_dataset, selected_indices_HT11)
    trial_loader = DataLoader(selected_set_HT11, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_6_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_6_dataset_trained_CNN1.pt: Spearman = 0.461093050507073
../../../CNN1_only_models/trial_1_best_6_dataset_trained_CNN1.pt: Spearman = 0.4315418183390163
../../../CNN1_only_models/trial_2_best_6_dataset_trained_CNN1.pt: Spearman = 0.43891886526443513
../../../CNN1_only_models/trial_3_best_6_dataset_trained_CNN1.pt: Spearman = 0.5414307409954462
../../../CNN1_only_models/trial_4_best_6_dataset_trained_CNN1.pt: Spearman = 0.5556875586800099
../../../CNN1_only_models/trial_5_best_6_dataset_trained_CNN1.pt: Spearman = 0.5079374015906246
../../../CNN1_only_models/trial_6_best_6_dataset_trained_CNN1.pt: Spearman = 0.5311053983589508
../../../CNN1_only_models/trial_7_best_6_dataset_trained_CNN1.pt: Spearman = 0.5843834259113322
../../../CNN1_only_models/trial_8_best_6_dataset_trained_CNN1.pt: Spearman = 0.5602233139181765
../../../CNN1_only_models/trial_9_best_6_dataset_trained_CNN1.pt: Spearman = 0.5877326565442536


In [13]:
# 7-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5)+CHANGE_seq+TIGER+iMeta)

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_HT11():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_HT11():
    with open('HT1-1_indel_frequency_value_percentage_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_7_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_HT11 = load_branch1_data_HT11()
    X1_HT11   = np.asarray(X1_HT11)
    X1 = np.concatenate([X1_HT11], axis=0) 

    rates_HT11 = load_reaction_rates_HT11()
    rates_HT11   = np.asarray(rates_HT11)
    rates = np.concatenate([rates_HT11], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen HT11 dataset
    np.random.seed(42)
    full_indices_HT11 = np.arange(len(rates_HT11))
    selected_indices_HT11 = np.random.choice(len(full_indices_HT11), size=len(full_indices_HT11), replace=False)
    unseen_indices_HT11 = np.setdiff1d(full_indices_HT11, selected_indices_HT11)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_HT11 = Subset(hybrid_dataset, unseen_indices_HT11)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_HT11 = Subset(hybrid_dataset, selected_indices_HT11)
    trial_loader = DataLoader(selected_set_HT11, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_7_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_7_dataset_trained_CNN1.pt: Spearman = 0.4647973961991489
../../../CNN1_only_models/trial_1_best_7_dataset_trained_CNN1.pt: Spearman = 0.4864476083992027
../../../CNN1_only_models/trial_2_best_7_dataset_trained_CNN1.pt: Spearman = 0.5841611453550057
../../../CNN1_only_models/trial_3_best_7_dataset_trained_CNN1.pt: Spearman = 0.47561668520256734
../../../CNN1_only_models/trial_4_best_7_dataset_trained_CNN1.pt: Spearman = 0.539163498946365
../../../CNN1_only_models/trial_5_best_7_dataset_trained_CNN1.pt: Spearman = 0.5676766519952773
../../../CNN1_only_models/trial_6_best_7_dataset_trained_CNN1.pt: Spearman = 0.5815455706297827
../../../CNN1_only_models/trial_7_best_7_dataset_trained_CNN1.pt: Spearman = 0.5878786765473019
../../../CNN1_only_models/trial_8_best_7_dataset_trained_CNN1.pt: Spearman = 0.5467030677069125
../../../CNN1_only_models/trial_9_best_7_dataset_trained_CNN1.pt: Spearman = 0.5344349881906173
